# 🎨 Data Designer Tutorial: Structured Outputs, Jinja Expressions, and Conditional Generation

#### 📚 What you'll learn

In this notebook, we will continue our exploration of Data Designer, demonstrating more advanced data generation using structured outputs, Jinja expressions, and conditional generation with `skip.when`.

If this is your first time using Data Designer, we recommend starting with the [first notebook](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/1-the-basics/) in this tutorial series.


### 📦 Import Data Designer

- `data_designer.config` provides access to the configuration API.

- `DataDesigner` is the main interface for data generation.


In [1]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

### ⚙️ Initialize the Data Designer interface

- `DataDesigner` is the main object that is used to interface with the library.

- When initialized without arguments, the [default model providers](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) are used.


In [2]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model that can be used during the generation process.

- The "model alias" is used to reference the model in the Data Designer config (as we will see below).

- The "model provider" is the external service that hosts the model (see the [model config](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) docs for more details).

- By default, we use [build.nvidia.com](https://build.nvidia.com/models) as the model provider.


In [3]:
# This name is set in the model provider configuration.
MODEL_PROVIDER = "nvidia"

# The model ID is from build.nvidia.com.
MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b"

# We choose this alias to be descriptive for our use case.
MODEL_ALIAS = "nemotron-nano-v3"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=1.0,
            top_p=1.0,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The Data Designer config defines the dataset schema and generation process.

- The config builder provides an intuitive interface for building this configuration.

- The list of model configs is provided to the builder at initialization.


In [4]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

### 🧑‍🎨 Designing our data

- We will again create a product review dataset, but this time we will use structured outputs and Jinja expressions.

- Structured outputs let you specify the exact schema of the data you want to generate.

- Data Designer supports schemas specified using either json schema or Pydantic data models (recommended).

<br>

We'll define our structured outputs using [Pydantic](https://docs.pydantic.dev/latest/) data models

> 💡 **Why Pydantic?**
>
> - Pydantic models provide better IDE support and type validation.
>
> - They are more Pythonic than raw JSON schemas.
>
> - They integrate seamlessly with Data Designer's structured output system.


In [5]:
from decimal import Decimal
from typing import Literal

from pydantic import BaseModel, Field


# We define a Product schema so that the name, description, and price are generated
# in one go, with the types and constraints specified.
class Product(BaseModel):
    name: str = Field(description="The name of the product")
    description: str = Field(description="A description of the product")
    price: Decimal = Field(description="The price of the product", ge=10, le=1000, decimal_places=2)


class ProductReview(BaseModel):
    rating: int = Field(description="The rating of the product", ge=1, le=5)
    customer_mood: Literal["irritated", "mad", "happy", "neutral", "excited"] = Field(
        description="The mood of the customer"
    )
    review: str = Field(description="A review of the product")

Next, let's design our product review dataset using a few more tricks compared to the previous notebook.


In [6]:
# Since we often only want a few attributes from Person objects, we can
# set drop=True in the column config to drop the column from the final dataset.
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="customer",
        sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
        params=dd.PersonFromFakerSamplerParams(),
        drop=True,
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="product_category",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Electronics",
                "Clothing",
                "Home & Kitchen",
                "Books",
                "Home Office",
            ],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="product_subcategory",
        sampler_type=dd.SamplerType.SUBCATEGORY,
        params=dd.SubcategorySamplerParams(
            category="product_category",
            values={
                "Electronics": [
                    "Smartphones",
                    "Laptops",
                    "Headphones",
                    "Cameras",
                    "Accessories",
                ],
                "Clothing": [
                    "Men's Clothing",
                    "Women's Clothing",
                    "Winter Coats",
                    "Activewear",
                    "Accessories",
                ],
                "Home & Kitchen": [
                    "Appliances",
                    "Cookware",
                    "Furniture",
                    "Decor",
                    "Organization",
                ],
                "Books": [
                    "Fiction",
                    "Non-Fiction",
                    "Self-Help",
                    "Textbooks",
                    "Classics",
                ],
                "Home Office": [
                    "Desks",
                    "Chairs",
                    "Storage",
                    "Office Supplies",
                    "Lighting",
                ],
            },
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="target_age_range",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(values=["18-25", "25-35", "35-50", "50-65", "65+"]),
    )
)

# Sampler columns support conditional params, which are used if the condition is met.
# In this example, we set the review style to rambling if the target age range is 18-25.
# Note conditional parameters are only supported for Sampler column types.
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="review_style",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["rambling", "brief", "detailed", "structured with bullet points"],
            weights=[1, 2, 2, 1],
        ),
        conditional_params={
            "target_age_range == '18-25'": dd.CategorySamplerParams(values=["rambling"]),
        },
    )
)

# Optionally validate that the columns are configured correctly.
data_designer.validate(config_builder)

[03:20:30] [INFO] ✅ Validation passed


Next, we will use more advanced Jinja expressions to create new columns.

Jinja expressions let you:

- Access nested attributes: `{{ customer.first_name }}`

- Combine values: `{{ customer.first_name }} {{ customer.last_name }}`

- Use conditional logic: `{% if condition %}...{% endif %}`


In [7]:
# We can create new columns using Jinja expressions that reference
# existing columns, including attributes of nested objects.
config_builder.add_column(
    dd.ExpressionColumnConfig(name="customer_name", expr="{{ customer.first_name }} {{ customer.last_name }}")
)

config_builder.add_column(dd.ExpressionColumnConfig(name="customer_age", expr="{{ customer.age }}"))

config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="product",
        prompt=(
            "Create a product in the '{{ product_category }}' category, focusing on products  "
            "related to '{{ product_subcategory }}'. The target age range of the ideal customer is "
            "{{ target_age_range }} years old. The product should be priced between $10 and $1000."
        ),
        output_format=Product,
        model_alias=MODEL_ALIAS,
    )
)

# We can even use if/else logic in our Jinja expressions to create more complex prompt patterns.
config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="customer_review",
        prompt=(
            "Your task is to write a review for the following product:\n\n"
            "Product Name: {{ product.name }}\n"
            "Product Description: {{ product.description }}\n"
            "Price: {{ product.price }}\n\n"
            "Imagine your name is {{ customer_name }} and you are from {{ customer.city }}, {{ customer.state }}. "
            "Write the review in a style that is '{{ review_style }}'."
            "{% if target_age_range == '18-25' %}"
            "Make sure the review is more informal and conversational.\n"
            "{% else %}"
            "Make sure the review is more formal and structured.\n"
            "{% endif %}"
            "The review field should contain only the review, no other text."
        ),
        output_format=ProductReview,
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)

[03:20:30] [INFO] ✅ Validation passed


## 🚦 Conditional generation with `skip.when`

So far, every column is generated for every row. But sometimes an expensive LLM column only makes sense
for a subset of rows — for example, a detailed complaint analysis is only useful when the review is negative.

Data Designer lets you **skip** column generation on a per-row basis using `SkipConfig`.
Skipped rows receive `None` by default, but you can provide a sentinel value with
`skip=dd.SkipConfig(when="...", value="N/A")` to write a specific value instead.

There are three patterns to know:

| Pattern | How | Effect |
|---|---|---|
| **Expression gate** | `skip=dd.SkipConfig(when="...")` | Skip this column when the Jinja2 expression is truthy |
| **Skip propagation** (default) | Downstream column depends on a skipped column | Automatically skipped too (`propagate_skip=True` by default) |
| **Propagation opt-out** | `propagate_skip=False` on the downstream column | Always generates, even if an upstream was skipped |


**Pattern 1 — Expression gate.** Only generate a detailed complaint analysis when the customer gave a low rating (1 or 2 stars).
Rows where the rating is 3 or higher will get `None` for this column.


In [8]:
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="complaint_analysis",
        model_alias=MODEL_ALIAS,
        prompt=(
            "A customer reviewed '{{ product.name }}' ({{ product_category }} / {{ product_subcategory }}).\n\n"
            "Review: {{ customer_review.review }}\n"
            "Rating: {{ customer_review.rating }}/5\n"
            "Mood: {{ customer_review.customer_mood }}\n\n"
            "Write a short root-cause analysis of why this customer is unhappy "
            "and suggest one concrete improvement the product team could make."
        ),
        skip=dd.SkipConfig(when="{{ customer_review.rating > 2 }}"),
    )
)

DataDesignerConfigBuilder(
    sampler_columns: [
        "customer",
        "product_category",
        "product_subcategory",
        "target_age_range",
        "review_style"
    ]
    llm_text_columns: ['complaint_analysis']
    llm_structured_columns: ['product', 'customer_review']
    expression_columns: ['customer_name', 'customer_age']
)

**Pattern 2 — Skip propagation.** `action_items` depends on `complaint_analysis`.
When `complaint_analysis` is skipped, `action_items` auto-skips too because
`propagate_skip` defaults to `True`.


In [9]:
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="action_items",
        model_alias=MODEL_ALIAS,
        prompt=(
            "Based on this complaint analysis:\n"
            "{{ complaint_analysis }}\n\n"
            "List 2-3 concrete action items for the product team."
        ),
    )
)

DataDesignerConfigBuilder(
    sampler_columns: [
        "customer",
        "product_category",
        "product_subcategory",
        "target_age_range",
        "review_style"
    ]
    llm_text_columns: ['complaint_analysis', 'action_items']
    llm_structured_columns: ['product', 'customer_review']
    expression_columns: ['customer_name', 'customer_age']
)

**Pattern 3 — Propagation opt-out.** `review_summary` also depends on `complaint_analysis`,
but sets `propagate_skip=False` so it always generates. The prompt uses a Jinja conditional
to handle the case where `complaint_analysis` is `None`.


In [10]:
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="review_summary",
        model_alias=MODEL_ALIAS,
        propagate_skip=False,
        prompt=(
            "Summarize this product review in one sentence:\n"
            "Product: {{ product.name }}\n"
            "Rating: {{ customer_review.rating }}/5\n"
            "Review: {{ customer_review.review }}\n"
            "{% if complaint_analysis %}"
            "Complaint analysis: {{ complaint_analysis }}\n"
            "{% endif %}"
        ),
    )
)

data_designer.validate(config_builder)

[03:20:30] [INFO] ✅ Validation passed


### 🔁 Iteration is key – preview the dataset!

1. Use the `preview` method to generate a sample of records quickly.

2. Inspect the results for quality and format issues.

3. Adjust column configurations, prompts, or parameters as needed.

4. Re-run the preview until satisfied.


In [11]:
preview = data_designer.preview(config_builder, num_records=2)

[03:20:30] [INFO] 🎥 Preview generation in progress


[03:20:30] [INFO]   |-- 🔒 Jinja rendering engine: secure


[03:20:30] [INFO] ✅ Validation passed


[03:20:30] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph


[03:20:30] [INFO] 🩺 Running health checks for models...


...

[03:20:36] [INFO]   |-- 🗂️ column: 'customer_review'


[03:20:36] [INFO]   |-- 📝 column: 'complaint_analysis'


[03:20:36] [INFO]   |-- 📝 column: 'action_items'


[03:20:36] [INFO]   |-- 📝 column: 'review_summary'


[03:20:36] [INFO] 🏆 Preview complete!


In [12]:
# Run this cell multiple times to cycle through the 2 preview records.
# Look for rows where complaint_analysis and action_items are None (skipped)
# vs rows where they were generated (low-rated reviews).
preview.display_sample_record()

                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                ┃ Value                                                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category    │ Books                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ product_subcategory │ Fiction                                                                              │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ target_age_range    │ 18-25                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ review_style        │ rambling                                                                             │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ complaint_analysis  │ None                                                                                 │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ action_items        │ None                                                                                 │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ review_summary      │ A vivid, poetic blend of interconnected turn‑of‑the‑century teen tales that captures │
│                     │ late‑night digital angst and wonder in a few coffee‑break‑length stories.            │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ product             │ {                                                                                    │
│                     │     'name': 'Midnight Musings: Turn-of-the-Century Teen Tales',                      │
│                     │     'description': "A collection of contemporary short fiction that captures the     │
│                     │ eclectic voice of today's 18-25 crowd, exploring themes of identity, digital         │
│                     │ alienation, and fleeting moments of wonder. Each story is a literary companion for   │
│                     │ late-night reading sessions, coffeehouse reflections, and classroom discussions,     │
│                     │ blending lyrical prose with relatable, modern narratives that resonate with emerging │
│                     │ adults.",                                                                            │
│                     │     'price': 24.99                                                                   │
│                     │ }                                                                                    │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ customer_review     │ {                                                                                    │
│                     │     'rating': 4,                                                                     │
│                     │     'customer_mood': 'happy',                                                        │
│                     │     'review': "Alrighty, so I just finished Midnight Musings and wow, what a ride.   │
│                     │ I’m sitting here in my little Lanefort apartment at 2 a.m., coffee half-cold, and I  │
│                     │ can't help but just... keep thinking about the stories. Like, the way the author     │
│   

In [13]:
# The preview dataset is available as a pandas DataFrame.
# Notice that complaint_analysis, action_items, and review_summary columns
# reflect the skip behavior: None for skipped rows, generated text otherwise.
preview.dataset

,product_category,product_subcategory,target_age_range,review_style,customer_name,customer_age,product,customer_review,complaint_analysis,action_items,review_summary
0,Books,Fiction,18-25,rambling,Brittany White,27,{'name': 'Midnight Musings: Turn-of-the-Centur...,"{'rating': 4, 'customer_mood': 'happy', 'revie...",None,None,"A vivid, poetic blend of interconnected turn‑o..."
1,Home Office,Lighting,50-65,structured with bullet points,Steven Russell,79,{'name': 'Tabletop Daylight 3000 Lux Adjustabl...,"{'rating': 5, 'customer_mood': 'happy', 'revie...",None,None,I highly recommend the Tabletop Daylight 3000 ...


### 📊 Analyze the generated data

- Data Designer automatically generates a basic statistical analysis of the generated data.

- This analysis is available via the `analysis` property of generation result objects.


In [14]:
# Print the analysis as a table.
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2                               │ 11                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                      ┃        data type ┃               number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category                 │           string │                         2 (100.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ product_subcategory              │           string │                         2 (100.0%) │          subcategory │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ target_age_range                 │           string │                         2 (100.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ review_style                     │           string │                         2 (100.0%) │             category │
└──────────────────────────────────┴──────────────────┴────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃                         ┃              ┃                           ┃       prompt tokens ┃    completion tokens ┃
┃ column name             ┃    data type ┃      number unique values ┃          per record ┃           per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ complaint_analysis      │         None │                  0 (0.0%) │     281.0 +/- 155.0 │          1.0 +/- 0.0 │
├─────────────────────────┼──────────────┼───────────────────────────┼─────────────────────┼──────────────────────┤
│ action_items            │         None │                  0 (0.0%) │        22.0 +/- 0.0 │          1.0 +/- 0.0 │
├─────────────────────────┼──────────────┼───────────────────────────┼─────────────────────┼──────────────────────┤
│ review_summary          │       string │                2 (100.0%) │     254.0 +/- 155.0 │        45.0 +/- 12.7 │
└─────────────────────────┴──────────────┴────────────────

### 🆙 Scale up!

- Happy with your preview data?

- Use the `create` method to submit larger Data Designer generation jobs.


In [15]:
results = data_designer.create(config_builder, num_records=10, dataset_name="tutorial-2")

[03:20:36] [INFO] 🎨 Creating Data Designer dataset


[03:20:36] [INFO]   |-- 🔒 Jinja rendering engine: secure


[03:20:36] [INFO] ✅ Validation passed


[03:20:36] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph


[03:20:36] [INFO] 🩺 Running health checks for models...


...

[03:20:48] [INFO]   |-- 🗂️ column: 'product'


[03:20:48] [INFO]   |-- 🗂️ column: 'customer_review'


[03:20:48] [INFO]   |-- 📝 column: 'complaint_analysis'


[03:20:48] [INFO]   |-- 📝 column: 'action_items'


[03:20:48] [INFO]   |-- 📝 column: 'review_summary'


In [16]:
# Load the generated dataset as a pandas DataFrame.
dataset = results.load_dataset()

dataset.head()

,product_category,product_subcategory,target_age_range,review_style,customer_name,customer_age,product,customer_review,complaint_analysis,action_items,review_summary
0,Home Office,Lighting,65+,detailed,Amanda Singh,41,"{'description': 'A sleek, touch-controlled LED...","{'customer_mood': 'happy', 'rating': 5, 'revie...",None,None,A 5‑star review praising the LuminaTouch LED T...
1,Home Office,Lighting,65+,rambling,Don Lopez,69,"{'description': 'A lightweight, ergonomic desk...","{'customer_mood': 'happy', 'rating': 5, 'revie...",None,None,I highly recommend the Ultra‑Soft Adjustable W...
2,Books,Textbooks,35-50,rambling,Benjamin Cisneros,30,{'description': 'A curated collection of found...,"{'customer_mood': 'happy', 'rating': 5, 'revie...",None,None,A five‑star review lauds this book as an elega...
3,Home Office,Storage,35-50,detailed,Brian Wise,92,"{'description': 'A sleek, space-saving storage...","{'customer_mood': 'happy', 'rating': 5, 'revie...",None,None,"The ErgoHome Foldable Desk Organizer, praised ..."
4,Electronics,Headphones,35-50,detailed,Matthew Collins,37,{'description': 'Premium wireless over-ear hea...,"{'customer_mood': 'excited', 'rating': 5, 'rev...",None,None,The reviewer lauds the HS‑350 headphones as a ...


In [17]:
# Load the analysis results into memory.
analysis = results.load_analysis()

analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 11                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                      ┃        data type ┃               number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category                 │           string │                          5 (50.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ product_subcategory              │           string │                          7 (70.0%) │          subcategory │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ target_age_range                 │           string │                          4 (40.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ review_style                     │           string │                          3 (30.0%) │             category │
└──────────────────────────────────┴──────────────────┴────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃                         ┃              ┃                            ┃      prompt tokens ┃    completion tokens ┃
┃ column name             ┃    data type ┃       number unique values ┃         per record ┃           per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ complaint_analysis      │         None │                   0 (0.0%) │     255.5 +/- 87.0 │          1.0 +/- 0.0 │
├─────────────────────────┼──────────────┼────────────────────────────┼────────────────────┼──────────────────────┤
│ action_items            │         None │                   0 (0.0%) │       22.0 +/- 0.0 │          1.0 +/- 0.0 │
├─────────────────────────┼──────────────┼────────────────────────────┼────────────────────┼──────────────────────┤
│ review_summary          │       string │                10 (100.0%) │     228.5 +/- 86.7 │         47.0 +/- 9.4 │
└─────────────────────────┴──────────────┴────────────────

## ⏭️ Next Steps

Check out the following notebook to learn more about:

- [Seeding synthetic data generation with an external dataset](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)

- [Providing images as context](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)

- [Generating images](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)
